In [ ]:
# 读取原始数据
import os
import csv
import pickle
import pandas as pd
from tqdm import tqdm

data = open("yago-meta-facts.ntx", "r")
fact_dict = {}
for line in data.readlines():
    if '<<' in line:
        fact, time = line.strip().split('<<')[1].split('>>')
        if 'schema' not in fact:
            continue
        sub, rel, obj = fact.strip().split('\t')
        sub = sub.strip().split('yago:')[1]
        sub = sub.replace('_', ' ')
        rel = rel.strip().split('schema:')[1]
        try:
            obj = obj.strip().split('yago:')[1]
            obj = obj.replace('_', ' ')
        except:
            continue
        
        try:
            time_type_tmp, time_year_temp = time.strip().split('Date\t"')
        except:
            continue
        time_type = time_type_tmp.strip().split(':')[1]
        time_year = time_year_temp.strip().split('-')[0]

        if (sub, rel, obj) not in fact_dict.keys():
            fact_dict[(sub, rel, obj)] = [-1,-1]
        
        if time_type == 'start':
            try:
                fact_dict[(sub, rel, obj)][0] = int(time_year)
                if int(time_year) > 2025:
                    fact_dict[(sub, rel, obj)][0] = -2
                if int(time_year) < 1801:
                    fact_dict[(sub, rel, obj)][0] = -2
            except:
                fact_dict[(sub, rel, obj)][0] = -2
        if time_type == 'end':
            try:
                fact_dict[(sub, rel, obj)][1] = int(time_year)
                if int(time_year) > 2025:
                    fact_dict[(sub, rel, obj)][1] = -2
                if int(time_year) < 1801:
                    fact_dict[(sub, rel, obj)][1] = -2
            except:
                fact_dict[(sub, rel, obj)][1] = -2

span_facts = {}
stamp_facts = {}        
for key in fact_dict.keys():
    if fact_dict[key][0] <= -1 and fact_dict[key][1] <= -1:
        continue
    else:
        if fact_dict[key][0] >= 0 and fact_dict[key][1] >= 0:
            if fact_dict[key][0] != fact_dict[key][1]:
                span_facts[key] = fact_dict[key]
            else:
                stamp_facts[key] = fact_dict[key][0]
        #else:
        #    stamp_facts[key] = max(fact_dict[key][0], fact_dict[key][1])
print(len(span_facts))
print(len(stamp_facts))

In [ ]:
span_entities = set()
span_relations = set()

stamp_entities = set()
stamp_relations = set()

span_min_time = 2000
span_max_time = -1

stamp_min_time = 2000
stamp_max_time = -1

for key in span_facts:
    sub, rel, obj = key
    span_entities.add(sub)
    span_entities.add(obj)
    span_relations.add(rel)

    if span_min_time > span_facts[key][0]:
        span_min_time = span_facts[key][0]
    
    if span_max_time < span_facts[key][1]:
        span_max_time = span_facts[key][1]

for key in stamp_facts:
    sub, rel, obj = key
    stamp_entities.add(sub)
    stamp_entities.add(obj)
    stamp_relations.add(rel)

    if stamp_min_time > stamp_facts[key]:
        stamp_min_time = stamp_facts[key]
    
    if stamp_max_time < stamp_facts[key]:
        stamp_max_time = stamp_facts[key]

print(len(span_entities))
print(len(stamp_entities))

print(len(span_relations))
print(len(stamp_relations))

print([span_min_time, span_max_time])
print([stamp_min_time, stamp_max_time])

In [ ]:
print(len(span_relations&stamp_relations))
print(len(span_relations|stamp_relations))
print(span_relations&stamp_relations)

In [ ]:
rel_num_dict = {}
for test_rel in list(span_relations|stamp_relations):
    span_num, stamp_num = 0, 0
    for key in span_facts:
        rel = key[1]
        if rel == test_rel:
            span_num += 1
    
    for key in stamp_facts:
        rel = key[1]
        if rel == test_rel:
            stamp_num += 1
    
    rel_num_dict[test_rel] = [span_num, stamp_num]
print(rel_num_dict)
        

In [ ]:
delete_rels = ['material', 'inLanguage', 'knowsLanguage', 'productionCompany', 'recordLabel', 'gender', 'superEvent', 'about', 'musicBy', 'lyricist'] # 筛掉相关知识数小于20或没有明显语义的relations
stamp_specific_rels = ['award', 'locationCreated', 'contentLocation', 'founder', 'author', 'alumniOf', 'birthPlace', 'deathPlace', 'children']# 对于原始数据集中时间戳与时间区间知识数量级差不多，且语义同时包含长期和瞬时的关系，保留原始知识；对于数量级相差较多的关系，检查其语义的长短期性质，并检查原始数据是否存在抽取错误，进而将其归纳为 瞬时知识（只用start time）
span_specific_rels = []
span_specific_rels_del = ['affiliation', 'organizer', 'performer', 'manufacturer', 'nationality', 'director', 'owns', 'actor', 'publisher', 'homeLocation', 'editor', 'spouse', 'worksFor', 'memberOf', 'location', 'sponsor'] #不确定是长期存在还是存在时间不到一年，因此直接删除这些关系的瞬时知识

new_span_facts = {}
new_stamp_facts = {}
for key in span_facts:
    sub, rel, obj = key
    if rel in delete_rels:
        continue
    if rel in stamp_specific_rels:
        if key not in new_stamp_facts.keys():
            new_stamp_facts[key] = span_facts[key][0]
    if rel in span_specific_rels:
        if key not in new_span_facts.keys():
            new_span_facts[key] = span_facts[key]
    if rel in span_specific_rels_del:
        if key not in new_span_facts.keys():
            new_span_facts[key] = span_facts[key]

for key in stamp_facts:
    sub, rel, obj = key
    if rel in delete_rels:
        continue
    if rel in stamp_specific_rels:
        if key not in new_stamp_facts.keys():
            new_stamp_facts[key] = stamp_facts[key]
    if rel in span_specific_rels:
        if key not in new_span_facts.keys():
            new_span_facts[key] = [stamp_facts[key], stamp_facts[key]+1]
    if rel in span_specific_rels_del:
        continue

print(len(new_span_facts))
print(len(new_stamp_facts))

In [ ]:
span_entities = set()
span_relations = set()

stamp_entities = set()
stamp_relations = set()

span_min_time = 2000
span_max_time = -1

stamp_min_time = 2000
stamp_max_time = -1

for key in new_span_facts:
    sub, rel, obj = key
    span_entities.add(sub)
    span_entities.add(obj)
    span_relations.add(rel)

    if span_min_time > new_span_facts[key][0]:
        span_min_time = new_span_facts[key][0]
    
    if span_max_time < new_span_facts[key][1]:
        span_max_time = new_span_facts[key][1]

for key in new_stamp_facts:
    sub, rel, obj = key
    stamp_entities.add(sub)
    stamp_entities.add(obj)
    stamp_relations.add(rel)

    if stamp_min_time > new_stamp_facts[key]:
        stamp_min_time = new_stamp_facts[key]
    
    if stamp_max_time < new_stamp_facts[key]:
        stamp_max_time = new_stamp_facts[key]

print(len(span_entities))
print(len(stamp_entities))

print(len(span_relations))
print(len(stamp_relations))

print([span_min_time, span_max_time])
print([stamp_min_time, stamp_max_time])

rel_num_dict = {}
for test_rel in list(span_relations|stamp_relations):
    span_num, stamp_num = 0, 0
    for key in new_span_facts:
        rel = key[1]
        if rel == test_rel:
            span_num += 1
    
    for key in new_stamp_facts:
        rel = key[1]
        if rel == test_rel:
            stamp_num += 1
    
    rel_num_dict[test_rel] = [span_num, stamp_num]
print(rel_num_dict)
        

In [ ]:
entity_2_facts = {}
has_span_entity = set()
stamp = set()
span = set()
min_time=2000
max_time=-1
for key in new_span_facts:
    sub = key[0]
    rel = key[1]
    obj = key[2]
    time = new_span_facts[key]
    if time[0] < min_time:
        min_time = time[0]
    if time[1] > max_time:
        max_time = time[1]

    if sub not in entity_2_facts.keys():
        entity_2_facts[sub] = set()
    entity_2_facts[sub].add((sub, rel, obj, time[0], time[1]))

    if obj not in entity_2_facts.keys():
        entity_2_facts[obj] = set()
    entity_2_facts[obj].add((sub, rel, obj, time[0], time[1]))

    has_span_entity.add(sub)
    has_span_entity.add(obj)
    span.add((sub, rel, obj, time[0], time[1]))

for key in new_stamp_facts:
    sub = key[0]
    rel = key[1]
    obj = key[2]
    time = new_stamp_facts[key]
    if time < min_time:
        min_time = time
    if time > max_time:
        max_time = time
    
    if sub not in entity_2_facts.keys():
        entity_2_facts[sub] = set()
    entity_2_facts[sub].add((sub, rel, obj, time))

    if obj not in entity_2_facts.keys():
        entity_2_facts[obj] = set()
    entity_2_facts[obj].add((sub, rel, obj, time))
    stamp.add((sub, rel, obj, time))

entity_2_facts_new = {}
for e in entity_2_facts.keys():
    fact_lens = [len(f) for f in entity_2_facts[e]]
    if 4 in fact_lens and 5 in fact_lens:
        entity_2_facts_new[e] = entity_2_facts[e]


print([min_time, max_time])
print(len(span))
print(len(stamp))
    


In [ ]:
import networkx as nx
G = nx.Graph() #nx.MultiGraph() #nx.Graph()
all_facts = set()
num = 0
for key in entity_2_facts_new.keys():
    for f in entity_2_facts_new[key]:
        #if key[1] == 'memberOf':
        #    continue
        s,o = f[0], f[2]
        G.add_edge(s, o)
        all_facts.add(f)
print(len(all_facts))
       
#max_g = G.subgraph(max(nx.connected_components(G), key=len))
max_core_number = nx.core_number(G)
max_k = max(max_core_number.values()) if max_core_number else 0
print(max_k)
max_g_1 = nx.k_core(G, k=max_k)
max_g_2 = nx.k_core(G, k=max_k-1)
max_g_3 = nx.k_core(G, k=max_k-2)
max_g_4 = nx.k_core(G, k=max_k-3)
max_g_5 = nx.k_core(G, k=max_k-4)
max_g_6 = nx.k_core(G, k=max_k-5)
max_g_7 = nx.k_core(G, k=max_k-6)
max_g_8 = nx.k_core(G, k=max_k-7)
max_g_9 = nx.k_core(G, k=max_k-8)
#print(len(max_g_1.nodes())+len(max_g_2.nodes()))
#print(len(max_g_1.edges())+len(max_g_2.edges()))

#3136178

In [ ]:
print(nx.is_bipartite(max_g_1))
print(nx.is_bipartite(max_g_2))
print(nx.is_bipartite(max_g_3))
print(nx.is_bipartite(max_g_4))
print(nx.is_bipartite(max_g_5))
print(nx.is_bipartite(max_g_6))
print(nx.is_bipartite(max_g_7))
print(nx.is_bipartite(max_g_8))
print(nx.is_bipartite(max_g_9))

In [ ]:
all_entities = set()
all_rels = set()
min_time = 2000
max_time = -1
span_knowledge = set()
stamp_knowledge = set()
all_nodes = list(max_g_1.nodes())
all_nodes.extend(list(max_g_2.nodes()))
all_nodes.extend(list(max_g_3.nodes()))
all_nodes.extend(list(max_g_4.nodes()))
all_nodes.extend(list(max_g_5.nodes()))
all_nodes.extend(list(max_g_6.nodes()))
#all_nodes.extend(list(max_g_8.nodes()))
#all_nodes.extend(list(max_g_9.nodes()))
print(list(entity_2_facts_new.keys())[:20])
print(list(all_nodes)[:20])

all_knowledge = set()
for f in tqdm.tqdm(all_facts):
    if f[0] in all_nodes and f[2] in all_nodes:
        all_knowledge.add(f)
   

print(len(all_knowledge))

In [ ]:
rel_span, rel_stamp = [], []
for r in all_rels:
    if r in stamp_specific_rels:
        rel_stamp.append(r)
    else:
        rel_span.append(r)
print(rel_span)
print(rel_stamp)

In [ ]:
time_2_num = {}
for fact in stamp_knowledge:
    s,r,o,t = fact
    if t not in time_2_num.keys():
        time_2_num[t] = 0
    time_2_num[t] += 1

for fact in span_knowledge:
    s,r,o,t_s,t_e = fact
    t_tmp = t_s
    while t_tmp <= t_e:
        if t_tmp not in time_2_num.keys():
            time_2_num[t_tmp] = 0
        time_2_num[t_tmp] += 1
        t_tmp += 1

time_num_list = sorted(time_2_num.items(), key=lambda x: x[1])
print(time_num_list[:1000])

In [ ]:
# check
facts = []
span_fact = set()
stamp_fact = set()
entities = set()
rels = set()
min_time = 2000
max_time = -1
for f in all_knowledge:
    if len(f) == 5:
        s,r,o,t_s,t_e = f
        entities.add(s)
        entities.add(o)
        rels.add(r)
        facts.append(f)

        if t_s < min_time:
            min_time=t_s
        if t_e > max_time:
            max_time=t_e
        span_fact.add(f)
    if len(f) == 4:
        s,r,o,t = f
        entities.add(s)
        entities.add(o)
        rels.add(r)
        facts.append(f)
        if t < min_time:
            min_time=t
        if t > max_time:
            max_time=t
        stamp_fact.add(f)

print(len(entities))
print(len(rels))
print(len(facts))
print([min_time, max_time])
print([len(span_fact), len(stamp_fact)])
print(facts[:20])

In [ ]:
pickle.dump(facts, open('facts_save_new_2.pkl', 'wb'))

In [ ]:
read_facts = pickle.load(open('facts_save_new_2.pkl', 'rb'))
print(len(read_facts))
print(read_facts[:20])

In [ ]:
saved_dict = pickle.load(open('e_2_keywords.pkl', 'rb'))
read_facts = pickle.load(open('facts_save_new_2.pkl', 'rb'))
all_nodes = set()
for f in read_facts:
    all_nodes.add(f[0])
    all_nodes.add(f[2])
print(len(saved_dict.keys()))
print(len(all_nodes))
print(len(set(saved_dict.keys())&all_nodes))

In [ ]:
import networkx as nx
G = nx.Graph() #nx.MultiGraph() #nx.Graph()
#entity_2_facts = {}
for fact in read_facts:
    s,r,o = fact[0], fact[1], fact[2]
    #if s not in saved_dict.keys() or o not in saved_dict.keys():
    #    continue
    #if s not in entity_2_facts.keys():
    #    entity_2_facts[s] = set()
    #entity_2_facts[s].add(tuple(fact))
    #if o not in entity_2_facts.keys():
    #    entity_2_facts[o] = set()
    #entity_2_facts[o].add(tuple(fact))
    G.add_edge(s, o)

max_core_number = nx.core_number(G)
max_k = max(max_core_number.values()) if max_core_number else 0
print(max_k)
max_g_1 = nx.k_core(G, k=max_k)
max_g_2 = nx.k_core(G, k=max_k-1)
max_g_3 = nx.k_core(G, k=max_k-2)
max_g_4 = nx.k_core(G, k=max_k-3)
max_g_5 = nx.k_core(G, k=max_k-4)
max_g_6 = nx.k_core(G, k=max_k-5)
max_g_7 = nx.k_core(G, k=max_k-6)
max_g_8 = nx.k_core(G, k=max_k-7)
max_g_9 = nx.k_core(G, k=max_k-8)

print(nx.is_bipartite(max_g_1))
print(nx.is_bipartite(max_g_2))
print(nx.is_bipartite(max_g_3))
print(nx.is_bipartite(max_g_4))
print(nx.is_bipartite(max_g_5))
print(nx.is_bipartite(max_g_6))
print(nx.is_bipartite(max_g_7))
print(nx.is_bipartite(max_g_8))
print(nx.is_bipartite(max_g_9))

In [ ]:
import tqdm
all_entities = set()
all_rels = set()
min_time = 2000
max_time = -1
span_knowledge = set()
stamp_knowledge = set()
all_nodes = list(max_g_2.nodes())
all_nodes.extend(list(max_g_3.nodes()))
all_nodes.extend(list(max_g_4.nodes()))
all_nodes.extend(list(max_g_5.nodes()))
all_nodes.extend(list(max_g_6.nodes()))
all_nodes.extend(list(max_g_6.nodes()))
#all_nodes.extend(list(max_g_7.nodes()))
#all_nodes.extend(list(max_g_8.nodes()))
#all_nodes.extend(list(max_g_9.nodes()))
for f in tqdm.tqdm(read_facts):
    if len(f) == 4: # stamp
        s,r,o,t = f
        #if len(span_knowledge)+len(stamp_knowledge) > 1000000:
        #if s not in all_nodes or o not in all_nodes:
        #    continue
        stamp_knowledge.add(f)
        if t < min_time:
            min_time = t
        if t > max_time:
            max_time = t
    if len(f) == 5: # span
        s,r,o,t_s,t_e = f
        #if len(span_knowledge)+len(stamp_knowledge) > 1000000:
        if s not in all_nodes or o not in all_nodes:
            continue
        span_knowledge.add(f)
        if t_s < min_time:
            min_time = t_s
        if t_e > max_time:
            max_time = t_e
    all_entities.add(s)
    all_entities.add(o)
    all_rels.add(r)

print(len(all_entities))
print(len(all_rels))
print(min_time)
print(max_time)
print(len(span_knowledge))
print(len(stamp_knowledge))


In [ ]:
read_facts = pickle.load(open('facts_save_new_2.pkl', 'rb'))
all_nodes_new = set()
for f in read_facts:
    all_nodes_new.add(f[0])
    all_nodes_new.add(f[2])

In [ ]:
# 按index查
import sys
import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint_url = "https://query.wikidata.org/sparql"
node_2_index_dict = {}

import re
def remove_matches(text):
    pattern2 = r'\bQ\d+\b'
    result = re.findall(pattern2, text)
    return result

def get_results(endpoint_url, query):
    user_agent = "WDQS-example Python/%s.%s" % (sys.version_info[0], sys.version_info[1])
    # TODO adjust user agent; see https://w.wiki/CX6
    sparql = SPARQLWrapper(endpoint_url, agent=user_agent)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.query().convert()

def batch_query_nodes(nodes, batch_size=10):
    """
    批量查询节点信息
    
    参数:
        nodes: 要查询的节点列表
        batch_size: 每批查询的节点数量
    """
    e_2_keywords = pickle.load(open('e_2_keywords.pkl', 'rb'))
    empty_nodes = pickle.load(open('empty_nodes.pkl', 'rb'))
    fail_nodes = pickle.load(open('fail_nodes.pkl', 'rb'))
    remain_nodes = list(set(nodes) - set(list(e_2_keywords.keys())))#- empty_nodes - fail_nodes)
    '''
    try:
        e_2_keywords = pickle.load(open('e_2_keywords.pkl', 'rb'))
        empty_nodes = pickle.load(open('empty_nodes.pkl', 'rb'))
        fail_nodes = pickle.load(open('fail_nodes.pkl', 'rb'))
        remain_nodes = list(set(nodes) - set(list(e_2_keywords.keys()))- empty_nodes - fail_nodes)
    except:
        e_2_keywords = {}
        remain_nodes = nodes
        pass
    '''
    # 分批处理节点
    for i in tqdm.tqdm(range(0, len(remain_nodes), batch_size)):
        batch_nodes = remain_nodes[i:i+batch_size]
        batch_query_nodes = []
        for node in batch_nodes:
            res = remove_matches(node)
            if len(res) != 0:
                batch_query_nodes.append(res[0])
                node_2_index_dict[node] = res[0]
        
         # 构建批量查询语句
        try:
            query = build_batch_query(batch_query_nodes)
            #print('here')
            results = get_results(endpoint_url, query)
            #print(results)
            
            # 处理批量结果
            process_batch_results(results, batch_nodes, e_2_keywords, empty_nodes)
        except:
            for node in batch_nodes:
                fail_nodes.add(node)
        print(len(e_2_keywords.keys()))
        pickle.dump(e_2_keywords, open('e_2_keywords.pkl', 'wb'))
        pickle.dump(empty_nodes, open('empty_nodes.pkl', 'wb'))
        pickle.dump(fail_nodes, open('fail_nodes.pkl', 'wb'))
        
        '''
        try:
            # 构建批量查询语句
            query = build_batch_query(batch_query_nodes)
            results = get_results(endpoint_url, query)
            for result in results["results"]["bindings"][:5]:
                print(result)
            
            # 处理批量结果
            process_batch_results(results, batch_nodes, e_2_keywords, empty_nodes)
            
        except Exception as e:
            print(f"批量查询失败: {str(e)}")
            for node in batch_nodes:
                fail_nodes.add(node)
        '''
    return e_2_keywords, empty_nodes, fail_nodes

def build_batch_query(query_nodes):
    """
    构建批量查询语句
    """
    # 创建VALUES子句
    values_clause = ' '.join([f"wd:{qid}" for qid in query_nodes])

    query = """
        SELECT ?person ?personLabel ?instanceOf ?instanceOfLabel ?occupationLabel WHERE {
            VALUES ?person { %s }
            OPTIONAL { ?person wdt:P106 ?occupation. }
            OPTIONAL { ?person wdt:P31 ?instanceOf. }
            SERVICE wikibase:label { 
                bd:serviceParam wikibase:language "en". 
            }
        }
        """%values_clause
    return query

def process_batch_results(results, batch_nodes, e_2_keywords, empty_nodes):
    """
    处理批量查询结果
    """
    
    # 按原始节点分组结果
    node_results = {}
    for result in results["results"]["bindings"]:
        original_node = result['person']['value']
        original_node = remove_matches(original_node)[0]
        
        if original_node not in node_results:
            node_results[original_node] = []
        
        node_results[original_node].append(result)
    
    # 为每个节点处理结果（限制前5个）
    for node in batch_nodes:
        if node in node_2_index_dict.keys() and node_2_index_dict[node] in node_results:
            results_for_node = node_results[node_2_index_dict[node]][:5]  # 限制每个节点最多5个结果
            
            for result in results_for_node:
                if 'instanceOfLabel' in result:
                    if node not in e_2_keywords.keys():
                        e_2_keywords[node] = set()
                    e_2_keywords[node].add(result['instanceOfLabel']['value'])
                if 'occupationLabel' in result:
                    if node not in e_2_keywords.keys():
                        e_2_keywords[node] = set()
                    e_2_keywords[node].add(result['occupationLabel']['value'])
        
        # 检查是否为空结果
        if node not in e_2_keywords.keys() or len(e_2_keywords[node]) == 0:
            empty_nodes.add(node)

# 使用示例
if __name__ == "__main__":
    # 假设 all_nodes 是你的节点列表
    query_nodes = list(all_nodes_new)
    
    # 批量查询（每批10个节点）
    e_2_keywords, tmp_empty_nodes, tmp_fail_nodes = batch_query_nodes(query_nodes, batch_size=500)

    
    # 打印结果
    print("查询结果:")
    for node, keywords in e_2_keywords.items():
        print(f"{node}: {keywords}")
    
    print(f"\n空结果节点: {tmp_empty_nodes}")
    print(f"失败节点: {tmp_fail_nodes}")

In [ ]:
# 按文本查
import sys
import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint_url = "https://query.wikidata.org/sparql"
node_2_query_dict = {}

import re
def remove_matches(text):
    pattern = r'[ _][A-Za-z]\d+[A-Z]?'
    result = re.sub(pattern, '', text)
    result = result.replace('_', ' ')
    result = re.sub(r"\s+", " ", result)
    return result

def get_results(endpoint_url, query):
    user_agent = "WDQS-example Python/%s.%s" % (sys.version_info[0], sys.version_info[1])
    # TODO adjust user agent; see https://w.wiki/CX6
    sparql = SPARQLWrapper(endpoint_url, agent=user_agent)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.query().convert()

def batch_query_nodes(nodes, batch_size=10):
    """
    批量查询节点信息
    
    参数:
        nodes: 要查询的节点列表
        batch_size: 每批查询的节点数量
    """
    e_2_keywords = pickle.load(open('e_2_keywords.pkl', 'rb'))
    empty_nodes = set()#pickle.load(open('empty_nodes.pkl', 'rb'))
    fail_nodes = set()#pickle.load(open('fail_nodes.pkl', 'rb'))
    remain_nodes = list(set(nodes) - set(list(e_2_keywords.keys())))
    
    # 分批处理节点
    for i in tqdm.tqdm(range(0, len(remain_nodes), batch_size)):
        batch_nodes = remain_nodes[i:i+batch_size]
        batch_query_nodes = []
        for node in batch_nodes:
            res = remove_matches(node)
            if len(res) != 0:
                batch_query_nodes.append(res)
                node_2_query_dict[node] = res

        # 构建批量查询语句
        query = build_batch_query(batch_query_nodes)
        results = get_results(endpoint_url, query)
        
        # 处理批量结果
        process_batch_results(results, batch_nodes, e_2_keywords, empty_nodes)
        print(len(e_2_keywords.keys()))
        pickle.dump(e_2_keywords, open('e_2_keywords.pkl', 'wb'))
        pickle.dump(empty_nodes, open('empty_nodes.pkl', 'wb'))
        pickle.dump(fail_nodes, open('fail_nodes.pkl', 'wb'))

        '''
        try:
            # 构建批量查询语句
            query = build_batch_query(batch_query_nodes)
            results = get_results(endpoint_url, query)
            
            # 处理批量结果
            process_batch_results(results, batch_nodes, e_2_keywords, empty_nodes)
            print(len(e_2_keywords.keys()))
            pickle.dump(e_2_keywords, open('e_2_keywords.pkl', 'wb'))
            pickle.dump(empty_nodes, open('empty_nodes.pkl', 'wb'))
            pickle.dump(fail_nodes, open('fail_nodes.pkl', 'wb'))
            
        except Exception as e:
            print(f"批量查询失败: {str(e)}")
            for node in batch_nodes:
                fail_nodes.add(node)
        '''
    
    return e_2_keywords, empty_nodes, fail_nodes

def build_batch_query(query_nodes):
    """
    构建批量查询语句
    """
    # 创建VALUES子句
    values_list = []
    for label in query_nodes:
        # 转义双引号并添加语言标签
        escaped_label = label.replace('"', '\\"')
        values_list.append(f'"{escaped_label}"@en')
    
    values_clause = " ".join(values_list)

    query = """SELECT ?person ?personLabel ?instanceOf ?instanceOfLabel ?occupationLabel WHERE {
    VALUES ?personLabel {%s}  # 根据需要添加更多标签
    ?person rdfs:label ?personLabel.
    FILTER(LANG(?personLabel) = "en")
    OPTIONAL { ?person wdt:P106 ?occupation. }
    OPTIONAL { ?person wdt:P31 ?instanceOf. }
    SERVICE wikibase:label { 
        bd:serviceParam wikibase:language "en". 
    }
    }"""%values_clause
    return query

def process_batch_results(results, batch_nodes, e_2_keywords, empty_nodes):
    """
    处理批量查询结果
    """
    # 初始化当前批次的节点
    
    # 按原始节点分组结果
    node_results = {}
    for result in results["results"]["bindings"]:
        original_node = result['personLabel']['value']
        
        if original_node not in node_results:
            node_results[original_node] = []
        
        node_results[original_node].append(result)
    
    #print(batch_nodes)
    #print(node_results.keys())
    # 为每个节点处理结果（限制前5个）
    for node in batch_nodes:
        if node in node_2_query_dict.keys() and node_2_query_dict[node] in node_results:
            results_for_node = node_results[node_2_query_dict[node]][:5]  # 限制每个节点最多5个结果
            
            for result in results_for_node:
                if 'instanceOfLabel' in result:
                    if node not in e_2_keywords.keys():
                        e_2_keywords[node] = set()
                    e_2_keywords[node].add(result['instanceOfLabel']['value'])
                if 'occupationLabel' in result:
                    if node not in e_2_keywords.keys():
                        e_2_keywords[node] = set()
                    e_2_keywords[node].add(result['occupationLabel']['value'])
        
        # 检查是否为空结果
        if node not in e_2_keywords.keys() or len(e_2_keywords[node]) == 0:
            empty_nodes.add(node)

# 使用示例
if __name__ == "__main__":
    # 假设 all_nodes 是你的节点列表
    query_nodes = list(all_nodes_new)
    
    # 批量查询（每批10个节点）
    e_2_keywords, empty_nodes, fail_nodes = batch_query_nodes(query_nodes, batch_size=100)
    
    # 打印结果
    print("查询结果:")
    for node, keywords in e_2_keywords.items():
        print(f"{node}: {keywords}")
    
    print(f"\n空结果节点: {empty_nodes}")
    print(f"失败节点: {fail_nodes}")

In [ ]:
final_node_2_keys = {}
final_facts = set()
span_num, stamp_num = 0, 0
nodes_has_keys = set(e_2_keywords.keys())
for f in tqdm.tqdm(read_facts): #tqdm.tqdm(stamp|span):
    if f[0] in nodes_has_keys and f[2] in nodes_has_keys and (f[0] in all_nodes or f[2] in all_nodes):
        final_facts.add(f)
        if len(f) < 4:
            print("error")
        if len(f) == 4:
            stamp_num += 1
        if len(f) == 5:
            span_num += 1
        final_node_2_keys[f[0]] = e_2_keywords[f[0]]
        final_node_2_keys[f[2]] = e_2_keywords[f[2]]

print(len(final_facts))
print(len(final_node_2_keys.keys()))
print([span_num, stamp_num])

In [ ]:
# check
e_2_keywords = pickle.load(open('e_2_keywords.pkl', 'rb'))
e_2_stamp_span = {}
r_2_fact = {}
t_2_fact = {}
for f in final_facts:
    if f[0] not in e_2_keywords.keys() and f[1] not in e_2_keywords.keys():
        print(f)
        break
    if len(f) == 4:
        s,r,o,t = f
        if s not in e_2_stamp_span.keys():
            e_2_stamp_span[s] = [set(),set()]
        e_2_stamp_span[s][0].add(f)

        if o not in e_2_stamp_span.keys():
            e_2_stamp_span[o] = [set(),set()]
        e_2_stamp_span[o][0].add(f)

        if r not in r_2_fact.keys():
            r_2_fact[r] = set()
        r_2_fact[r].add(f)

        if t not in t_2_fact.keys():
            t_2_fact[t] = set()
        t_2_fact[t].add(f)
    
    if len(f) == 5:
        s,r,o,t_s, t_e = f
        if s not in e_2_stamp_span.keys():
            e_2_stamp_span[s] = [set(),set()]
        e_2_stamp_span[s][1].add(f)

        if o not in e_2_stamp_span.keys():
            e_2_stamp_span[o] = [set(),set()]
        e_2_stamp_span[o][1].add(f)

        if r not in r_2_fact.keys():
            r_2_fact[r] = set()
        r_2_fact[r].add(f)

        for t in range(t_s, t_e):
            if t not in t_2_fact.keys():
                t_2_fact[t] = set()
            t_2_fact[t].add(f)

e_no_span_or_stamp = set()
for e in e_2_stamp_span.keys():
    if len(e_2_stamp_span[e][0]) == 0 or len(e_2_stamp_span[e][1]) == 0:
        e_no_span_or_stamp.add(e)
print([len(e_2_stamp_span.keys()), len(e_no_span_or_stamp)])
print([(key,len(r_2_fact[key])) for key in r_2_fact.keys()])

print(len(final_facts))

min_num = 1000
max_num = -1
for t in t_2_fact.keys():
    num = len(t_2_fact[t])
    if num > max_num:
        max_num = num
    if num < min_num:
        min_num = num
print([max_num, min_num])

In [ ]:
# 去除实体文本中的乱码
import re
def remove_matches(text):
    pattern = r'[ _][A-Za-z]\d+[A-Z]?'
    result = re.sub(pattern, '', text)
    result = result.replace('_', ' ')
    result = re.sub(r"\s+", " ", result)
    return result

save_node_2_keys={}
save_fact=set()
for f in final_facts:
    if len(f) == 4:
        save_fact.add((remove_matches(f[0]), f[1], remove_matches(f[2]), f[3]))
    if len(f) == 5:
        save_fact.add((remove_matches(f[0]), f[1], remove_matches(f[2]), f[3], f[4]))

for e in e_2_keywords.keys():
    save_node_2_keys[remove_matches(e)] = e_2_keywords[e]



e_2_stamp_span = {}
r_2_fact = {}
t_2_fact = {}
for f in save_fact:
    if f[0] not in save_node_2_keys.keys() and f[1] not in save_node_2_keys.keys():
        print(f)
        break
    if len(f) == 4:
        s,r,o,t = f
        if s not in e_2_stamp_span.keys():
            e_2_stamp_span[s] = [set(),set()]
        e_2_stamp_span[s][0].add(f)

        if o not in e_2_stamp_span.keys():
            e_2_stamp_span[o] = [set(),set()]
        e_2_stamp_span[o][0].add(f)

        if r not in r_2_fact.keys():
            r_2_fact[r] = set()
        r_2_fact[r].add(f)

        if t not in t_2_fact.keys():
            t_2_fact[t] = set()
        t_2_fact[t].add(f)
    
    if len(f) == 5:
        s,r,o,t_s, t_e = f
        if s not in e_2_stamp_span.keys():
            e_2_stamp_span[s] = [set(),set()]
        e_2_stamp_span[s][1].add(f)

        if o not in e_2_stamp_span.keys():
            e_2_stamp_span[o] = [set(),set()]
        e_2_stamp_span[o][1].add(f)

        if r not in r_2_fact.keys():
            r_2_fact[r] = set()
        r_2_fact[r].add(f)

        for t in range(t_s, t_e):
            if t not in t_2_fact.keys():
                t_2_fact[t] = set()
            t_2_fact[t].add(f)

e_num_list = []
e_no_span_or_stamp = set()
for e in e_2_stamp_span.keys():
    if len(e_2_stamp_span[e][0]) == 0 or len(e_2_stamp_span[e][1]) == 0:
        e_no_span_or_stamp.add(e)
    e_num_list.append(len(e_2_stamp_span[e][0])+len(e_2_stamp_span[e][1]))
    #if num > max_e_num:
    #    max_e_num = num
    #if num < min_e_num:
    #    min_e_num = num

e_num_list = sorted(e_num_list)
print(e_num_list[1000])
print([len(e_2_stamp_span.keys()), len(e_no_span_or_stamp)])
print([(key,len(r_2_fact[key])) for key in r_2_fact.keys()])
print(len(save_fact))

min_num = 1000
max_num = -1
for t in t_2_fact.keys():
    num = len(t_2_fact[t])
    if num > max_num:
        max_num = num
    if num < min_num:
        min_num = num
print([max_num, min_num])

In [ ]:
print(e_num_list[49180])
print(r_2_fact.keys())

rel_span, rel_stamp = [], []
for r in r_2_fact.keys():
    if r in stamp_specific_rels:
        rel_stamp.append(r)
    else:
        rel_span.append(r)
print(rel_span)
print(rel_stamp)


In [ ]:
pickle.dump([save_node_2_keys, save_fact], open('YAGO_e40k_f130k.pkl', 'wb'))

In [ ]:
import pickle
node_2_keys, fact = pickle.load(open('YAGO_e40k_f130k.pkl', 'rb'))
print([(key, node_2_keys[key]) for key in list(node_2_keys.keys())[:30]])
print(list(fact)[:30])

In [ ]:
import matplotlib.pyplot as plt
import tqdm
# split train valid test
# 首先确定切分时间戳
all_times = set()
t_2_stamp_fact = {}
t_2_span_fact = {}
t_2_start = {}
t_2_end = {}
new_fact = set()
for f in tqdm.tqdm(fact):
    if len(f) == 4:
        all_times.add(f[-1])
        if f[-1] not in t_2_stamp_fact.keys():
            t_2_stamp_fact[f[-1]] = set()
        t_2_stamp_fact[f[-1]].add(f)
        new_fact.add(f)
    if len(f) == 5:
        if f[-1] <= f[-2]:
            continue
        all_times.add(f[-1])
        all_times.add(f[-2])
        for t in range(f[-2], f[-1]+1):
            if t not in t_2_span_fact.keys():
                t_2_span_fact[t] = set()
            t_2_span_fact[t].add(f)
        
        if f[-2] not in t_2_start.keys():
            t_2_start[f[-2]] = set()
        t_2_start[f[-2]].add(f)

        if f[-1] not in t_2_end.keys():
            t_2_end[f[-1]] = set()
        t_2_end[f[-1]].add(f)
        new_fact.add(f)


sorted_all_times = sorted(list(all_times))
sorted_stamp_nums = []
sorted_span_nums = []
sorted_start_nums = []
sorted_end_nums = []
for t in sorted_all_times:
    if t in t_2_stamp_fact.keys():
        sorted_stamp_nums.append(len(t_2_stamp_fact[t]))
    else:
        sorted_stamp_nums.append(0)
    
    if t in t_2_span_fact.keys():
        sorted_span_nums.append(len(t_2_span_fact[t]))
    else:
        sorted_span_nums.append(0)

    if t in t_2_start.keys():
        sorted_start_nums.append(len(t_2_start[t]))
    else:
        sorted_start_nums.append(0)
    
    if t in t_2_end.keys():
        sorted_end_nums.append(len(t_2_end[t]))
    else:
        sorted_end_nums.append(0)
print(sorted_all_times)
print(len(sorted_all_times))
print(sorted_stamp_nums)
print(len(sorted_stamp_nums))
print(sorted_span_nums)
print(len(sorted_span_nums))

split_ratio = 0.9 # 0.9 for e40k_f130k
valid_preiod = 0.025 # 0.025 for e40k_f130k
# 切分时间戳
print('train/valid/test split point')
print([int(split_ratio*len(sorted_all_times)), sorted_all_times[int(split_ratio*len(sorted_all_times))]])
print([int((split_ratio+valid_preiod)*len(sorted_all_times)), sorted_all_times[int((split_ratio+valid_preiod)*len(sorted_all_times))]])

print('train stamp span fact num')
# 切分后的训练集时间戳知识数量
print(sum(sorted_stamp_nums[:int(split_ratio*len(sorted_all_times))]))
# 切分后的训练集时间区间知识数量
print(sum(sorted_span_nums[:int(split_ratio*len(sorted_all_times))]))

print('valid stamp span fact num')
# 切分后的验证集时间戳知识数量
print(sum(sorted_stamp_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))
# 切分后的验证集时间区间知识数量
print(sum(sorted_span_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))

print('test stamp span fact num')
# 切分后的测试集时间戳知识数量
print(sum(sorted_stamp_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):]))
# 切分后的测试集时间区间知识数量
print(sum(sorted_span_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):]))

# 检查验证集和测试集各有多少时间区间知识结束和时间区间知识开始
print('train start end fact num')
print(sum(sorted_start_nums[:int(split_ratio*len(sorted_all_times))]))
print(sum(sorted_end_nums[:int(split_ratio*len(sorted_all_times))]))
print('valid start end fact num')
print(sum(sorted_start_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))
print(sum(sorted_end_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))

print('test start end fact num')
print(sum(sorted_start_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):-1]))
print(sum(sorted_end_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):-1]))

plt.plot(sorted_stamp_nums, color='skyblue', alpha=0.5)
plt.plot(sorted_span_nums, color='red', alpha=0.5)
plt.show()

In [ ]:
# 正式切分数据集
train_set = set()
valid_set = set()
test_set = set()

train_split_time = sorted_all_times[int(split_ratio*len(sorted_all_times))-1]
valid_split_time = sorted_all_times[int((split_ratio+valid_preiod)*len(sorted_all_times))-1]
for f in new_fact:
    if len(f) == 4: # stamp knowledge
        s, r, o, t = f
        if t <= train_split_time:
            train_set.add(f)
        elif t <= valid_split_time:
            valid_set.add(f)
        else:
            test_set.add(f)
    if len(f) == 5: # span knowledge
        s, r, o, t_s, t_o = f
        if t_s <= train_split_time:
            train_set.add(f)
        elif t_s <= valid_split_time:
            valid_set.add(f)
        else:
            test_set.add(f)
        
        if t_o > train_split_time:
            valid_set.add(f)
        if t_o > valid_split_time:
            test_set.add(f)

print([len(train_set), len(valid_set), len(test_set)])
print(len(train_set|valid_set|test_set))

# check the consistency
print('train stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in train_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in train_set if len(f) == 5]))

print('valid stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in valid_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in valid_set if len(f) == 5]))

print('test stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in test_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in test_set if len(f) == 5]))

# 检查验证集和测试集各有多少时间区间知识结束和时间区间知识开始
print('train start end fact num')
start_num, end_num = 0, 0
train_start_fact = set()
train_end_fact = set()
for f in train_set:
    if len(f) == 5:
        if f[-2] <= train_split_time:
            start_num += 1
            train_start_fact.add(f)
        if f[-1] <= train_split_time:
            end_num += 1
            train_end_fact.add(f)
print([start_num, end_num])

print('valid start end fact num')
start_num, end_num = 0, 0
valid_start_fact = set()
valid_end_fact = set()
for f in valid_set:
    if len(f) == 5:
        if f[-2] <= valid_split_time and f[-2] > train_split_time:
            start_num += 1
            valid_start_fact.add(f)
        if f[-1] <= valid_split_time and f[-1] > train_split_time:
            end_num += 1
            valid_end_fact.add(f)          
print([start_num, end_num])

print('test start end fact num')
start_num, end_num = 0, 0
test_start_fact = set()
test_end_fact = set()
for f in test_set:
    if len(f) == 5:
        if f[-2] > valid_split_time:
            start_num += 1
            test_start_fact.add(f)
        if f[-1] < sorted_all_times[-1] and f[-1] > valid_split_time:
            end_num += 1
            test_end_fact.add(f)
print([start_num, end_num])

# check train_valid_test 泄漏
# stamp knowledge
train_stamp_facts = set([f for f in train_set if len(f) == 4])
valid_stamp_facts = set([f for f in valid_set if len(f) == 4])
test_stamp_facts = set([f for f in test_set if len(f) == 4])

print("stamp train & valid")
print(len(train_stamp_facts&valid_stamp_facts))

print("stamp valid & test")
print(len(test_stamp_facts&valid_stamp_facts))

print("stamp train & test")
print(len(train_stamp_facts&test_stamp_facts))

# span knowledge
print("start train & valid")
print(len(train_start_fact&valid_start_fact))

print("start train & test")
print(len(train_start_fact&test_start_fact))

print("start test & valid")
print(len(test_start_fact&valid_start_fact))

print("end train & valid")
print(len(train_end_fact&valid_end_fact))

print("end train & test")
print(len(train_end_fact&test_end_fact))

print("end test & valid")
print(len(test_end_fact&valid_end_fact))

In [ ]:
pickle.dump([train_split_time, valid_split_time, train_set, valid_set, test_set, node_2_keys], open('YAGO_e40k_f130k_train_valid_test.pkl', 'wb'))